# 03 - Xây dựng mô hình

Giai đoạn 4 của kế hoạch: huấn luyện 3 mô hình:
- **Logistic Regression** (Spark MLlib) - baseline, dễ giải thích
- **Random Forest** (Spark MLlib) - xử lý tốt phi tuyến
- **Isolation Forest** (scikit-learn) - unsupervised, không cần nhãn khi huấn luyện

**Yêu cầu**: đã chạy `python src/spark_etl.py` và `python src/imbalance.py` trước đó
(hoặc `python run_pipeline.py`) để có `data/processed/train_weighted.parquet` và `test.parquet`.

Notebook này gọi trực tiếp các hàm trong `src/train_models.py`.

In [1]:
import os
import sys

os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot")
os.environ.setdefault("HADOOP_HOME", r"D:\fraud-detection\hadoop")
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

sys.path.insert(0, os.path.abspath("../src"))

from config import MODELS_DIR, TRAIN_WEIGHTED_PARQUET_PATH
from spark_etl import get_spark_session
from train_models import (
    assemble_features,
    train_logistic_regression,
    train_random_forest,
    train_isolation_forest,
)

spark = get_spark_session("Modeling-Notebook")
train_df = spark.read.parquet(str(TRAIN_WEIGHTED_PARQUET_PATH))
train_df = assemble_features(train_df)
train_df.cache()
print(f"Số dòng train: {train_df.count():,}")

Số dòng train: 198,402


## Logistic Regression

In [2]:
lr_model = train_logistic_regression(train_df)
print("Hệ số (coefficients):", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

[Train] Huấn luyện Logistic Regression (Spark MLlib)...


Hệ số (coefficients): [-0.12843140426408864,0.3637851826044275,-0.017988629628677418,0.11731773606805264,-0.05918149583725937,0.4395264287251516,0.03603512544141027,-0.20751473083026648,-0.017347168895126563,-0.11181169317929765,-0.09668352375464004,-0.14427269393968695,0.1839213445182053,-0.20178448512738423,-0.23857019298192614,-0.2983659666127762,-0.0137786224768613,-0.1083095419061306,-0.009946285268707113,0.04922020855385131,-0.06052906325555499,-0.15253165303115074,0.08076191763414,0.2287599913816736,-0.07873154756487753,-0.29886997422847555,0.07182537645861699,-0.09729546322168858,0.03427499683341162,0.1404714093207173]
Intercept: -2.314748177490162


## Random Forest

Xem thêm mức độ quan trọng của từng đặc trưng (feature importance).

In [3]:
import pandas as pd
from config import MODEL_FEATURE_COLS

rf_model = train_random_forest(train_df)
importances = pd.Series(rf_model.featureImportances.toArray(), index=MODEL_FEATURE_COLS)
importances.sort_values(ascending=False).head(10)

[Train] Huấn luyện Random Forest (Spark MLlib)...


V14    0.192027
V10    0.168173
V4     0.125549
V17    0.108496
V12    0.095427
V11    0.057846
V16    0.043803
V3     0.040028
V21    0.017157
V7     0.016647
dtype: float64

## Isolation Forest (unsupervised)

Huấn luyện KHÔNG dùng cột `Class` - chỉ dùng nhãn thật khi đánh giá ở notebook 04.

In [4]:
iso_model = train_isolation_forest(train_df)
iso_model

[Train] Huấn luyện Isolation Forest (scikit-learn, unsupervised)...


,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


## Lưu mô hình

In [5]:
import joblib

lr_model.write().overwrite().save(str(MODELS_DIR / "logistic_regression"))
rf_model.write().overwrite().save(str(MODELS_DIR / "random_forest"))
joblib.dump(iso_model, MODELS_DIR / "isolation_forest.joblib")
print("Đã lưu cả 3 mô hình vào", MODELS_DIR)

Đã lưu cả 3 mô hình vào D:\fraud-detection\models


## (Tùy chọn) Tinh chỉnh siêu tham số

Grid Search + Cross-Validation, so sánh class-weight vs SMOTE, và tối ưu ngưỡng phân loại
được cài trong `src/tune_models.py` - chạy tách riêng vì tốn nhiều thời gian hơn:

```bash
python src/tune_models.py
```

In [6]:
spark.stop()